# Paddy Doctor classifier training (GPU)

This notebook trains the same 10-class, class-balanced PaddyDoctorCNN used by the API. It uses only the labelled `train_images/<class>/` folders; it does not use the separate unlabeled `test_images` folder. A stratified 80/20 image-level split is used (seed 42).

**Colab setup:** connect this notebook to a Colab GPU runtime in VS Code, then put `train_images.zip` (or `train_images.zipp`) under `MyDrive/Paddyvision_AI_Dataset/05_Pest_and_Disease_Data/Paddy_Doctor/`. The setup cell extracts it before training. The notebook refuses to start if CUDA is not available. Mounting Drive prompts Google authentication in the runtime.

In [2]:
# Runtime + data setup
from pathlib import Path
import os, random, json, time
from collections import Counter
import numpy as np
import torch
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is not connected. In VS Code, connect this notebook to a Colab GPU runtime and rerun this cell.')
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/Paddyvision_AI_Dataset/05_Pest_and_Disease_Data/Paddy_Doctor')
    DATA_DIR = DATA_ROOT / 'train_images'
    OUTPUT_DIR = Path('/content/drive/MyDrive/Paddyvision_AI_Dataset/training_outputs/paddy_doctor')
except ImportError:
    # For a non-Colab Jupyter kernel, edit these two paths to local folders.
    DATA_ROOT = Path(r'C:/Paddyvision_AI_Dataset/05_Pest_and_Disease_Data/Paddy_Doctor')
    DATA_DIR = DATA_ROOT / 'train_images'
    OUTPUT_DIR = Path(r'C:/Users/adnan/PaddyVision-AI/models')

if not DATA_DIR.is_dir():
    archives = [DATA_ROOT / 'train_images.zip', DATA_ROOT / 'train_images.zipp']
    archive_path = next((path for path in archives if path.is_file()), None)
    if archive_path is None:
        expected = ', '.join(str(path) for path in archives)
        raise FileNotFoundError(f'Training folder not found at {DATA_DIR}, and no archive was found. Expected one of: {expected}')
    from zipfile import ZipFile
    print(f'Extracting {archive_path} ...')
    with ZipFile(archive_path) as archive:
        archive.extractall(DATA_ROOT)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Archive was extracted, but {DATA_DIR} was not created. Check that the archive contains a train_images/ folder.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Training data:', DATA_DIR)
print('Outputs:', OUTPUT_DIR)

GPU: Tesla T4
PyTorch: 2.11.0+cu130 | CUDA: 13.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting /content/drive/MyDrive/Paddyvision_AI_Dataset/05_Pest_and_Disease_Data/Paddy_Doctor/train_images.zip ...
Training data: /content/drive/MyDrive/Paddyvision_AI_Dataset/05_Pest_and_Disease_Data/Paddy_Doctor/train_images
Outputs: /content/drive/MyDrive/Paddyvision_AI_Dataset/training_outputs/paddy_doctor


In [3]:
from pathlib import Path
from zipfile import ZipFile

base = Path("/content/drive/MyDrive/Paddyvision_AI_Dataset/05_Pest_and_Disease_Data/Paddy_Doctor")

with ZipFile(base / "train_images.zip") as archive:
    archive.extractall(base)

print("Training folder found:", (base / "train_images").is_dir())

Training folder found: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Model architecture — same as api/services/paddy_doctor_model.py
import torch.nn as nn
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x): return self.block(x)
class PaddyDoctorCNN(nn.Module):
    def __init__(self, class_count):
        super().__init__()
        self.features = nn.Sequential(ConvBlock(3,24), ConvBlock(24,48), ConvBlock(48,96), ConvBlock(96,128), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = nn.Sequential(nn.Dropout(0.25), nn.Linear(128,class_count))
    def forward(self,x): return self.classifier(self.features(x))
print('Model class ready')

Model class ready


In [6]:
# Stratified 80/20 split, image loading, and validation metrics
SEED, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 160, 32, 8
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
CLASSES = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
assert len(CLASSES) >= 2, f'Expected class folders in {DATA_DIR}'
train_samples, val_samples = [], []
extensions = {'.jpg','.jpeg','.png','.webp'}
for label, name in enumerate(CLASSES):
    files = sorted(p for p in (DATA_DIR/name).iterdir() if p.suffix.lower() in extensions)
    if len(files) < 2: raise ValueError(f'{name} needs at least two images')
    random.Random(SEED+label).shuffle(files)
    n_val = max(1, round(0.2*len(files)))
    val_samples += [(p,label) for p in files[:n_val]]
    train_samples += [(p,label) for p in files[n_val:]]
print('Classes:', len(CLASSES), CLASSES)
print('Train:', len(train_samples), '| validation:', len(val_samples))

class RiceLeafDataset(Dataset):
    def __init__(self,samples,augment=False): self.samples,self.augment=samples,augment
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        path,label=self.samples[i]
        with Image.open(path) as im:
            im=im.convert('RGB')
            if self.augment and random.random()<0.5: im=ImageOps.mirror(im)
            im=im.resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
            arr=np.asarray(im,dtype=np.float32)/255.0
        return torch.from_numpy(arr).permute(2,0,1),label
train_loader=DataLoader(RiceLeafDataset(train_samples,True),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(RiceLeafDataset(val_samples),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)

def evaluate(model):
    matrix=torch.zeros((len(CLASSES),len(CLASSES)),dtype=torch.int64)
    model.eval()
    with torch.inference_mode():
        for x,y in val_loader:
            pred=model(x.to(device,non_blocking=True)).argmax(1).cpu()
            for a,p in zip(y,pred): matrix[int(a),int(p)]+=1
    per_class={}; f1s=[]
    for i,name in enumerate(CLASSES):
        tp=int(matrix[i,i]); fp=int(matrix[:,i].sum())-tp; fn=int(matrix[i,:].sum())-tp
        precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.; f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        per_class[name]={'precision':precision,'recall':recall,'f1':f1,'support':int(matrix[i].sum())}; f1s.append(f1)
    return {'accuracy':float(matrix.diag().sum())/int(matrix.sum()),'macro_f1':sum(f1s)/len(f1s),'per_class':per_class,'confusion_matrix':matrix.tolist(),'validation_image_count':int(matrix.sum())}
print('Split and loaders ready')

Classes: 10 ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']
Train: 8325 | validation: 2082
Split and loaders ready


In [7]:
# Train with CUDA; save best validation macro-F1 checkpoint and report
model=PaddyDoctorCNN(len(CLASSES)).to(device)
freq=Counter(y for _,y in train_samples)
weights=torch.tensor([len(train_samples)/(len(CLASSES)*freq[i]) for i in range(len(CLASSES))],dtype=torch.float32,device=device)
criterion=nn.CrossEntropyLoss(weight=weights)
optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=1e-4)
checkpoint_path=OUTPUT_DIR/'paddy_doctor_classifier.pth'
report_path=OUTPUT_DIR/'paddy_doctor_validation.json'
best_f1=-1.; stale=0; patience=3; history=[]
for epoch in range(EPOCHS):
    model.train(); loss_sum=0.; start=time.time()
    for x,y in train_loader:
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True); loss=criterion(model(x),y); loss.backward(); optimizer.step()
        loss_sum += float(loss.detach())*len(y)
    report=evaluate(model); report['epoch']=epoch+1; report['train_loss']=loss_sum/len(train_samples)
    history.append({'epoch':epoch+1,'train_loss':report['train_loss'],'accuracy':report['accuracy'],'macro_f1':report['macro_f1']})
    print(f"epoch {epoch+1}/{EPOCHS} | loss {report['train_loss']:.4f} | val acc {report['accuracy']:.4f} | macro F1 {report['macro_f1']:.4f} | {time.time()-start:.1f}s",flush=True)
    if report['macro_f1']>best_f1:
        best_f1=report['macro_f1']; stale=0
        torch.save({'state_dict':model.state_dict(),'class_names':CLASSES,'image_size':IMAGE_SIZE,'training_dataset':'Paddy Doctor external image dataset','validation_split':'stratified 80/20 from labelled train_images; seed 42'},checkpoint_path)
        report.update({'dataset':'Paddy Doctor external image dataset','class_names':CLASSES,'train_image_count':len(train_samples),'split_seed':SEED,'split':'stratified 80/20 from labelled training folders','external_test_images_used':False,'limitations':'Image-level holdout only; near-duplicate views may cross the split. Scores do not estimate generalization to new farms or Sekinchan field conditions.'})
        report_path.write_text(json.dumps(report,indent=2),encoding='utf-8')
    else:
        stale+=1
        if stale>=patience: print('Early stopping: validation macro F1 did not improve.'); break
print('Best validation macro F1:',round(best_f1,4))
print('Checkpoint:',checkpoint_path)
print('Report:',report_path)
print('History:',history)

epoch 1/8 | loss 2.0100 | val acc 0.2493 | macro F1 0.2166 | 64.2s
epoch 2/8 | loss 1.7330 | val acc 0.2459 | macro F1 0.1800 | 59.6s
epoch 3/8 | loss 1.5422 | val acc 0.3425 | macro F1 0.3225 | 59.9s
epoch 4/8 | loss 1.3704 | val acc 0.4126 | macro F1 0.3675 | 59.6s
epoch 5/8 | loss 1.2465 | val acc 0.5889 | macro F1 0.5750 | 60.2s
epoch 6/8 | loss 1.1560 | val acc 0.6071 | macro F1 0.5919 | 59.1s
epoch 7/8 | loss 1.0510 | val acc 0.6066 | macro F1 0.6026 | 61.4s
epoch 8/8 | loss 0.9852 | val acc 0.6671 | macro F1 0.6601 | 59.5s
Best validation macro F1: 0.6601
Checkpoint: /content/drive/MyDrive/Paddyvision_AI_Dataset/training_outputs/paddy_doctor/paddy_doctor_classifier.pth
Report: /content/drive/MyDrive/Paddyvision_AI_Dataset/training_outputs/paddy_doctor/paddy_doctor_validation.json
History: [{'epoch': 1, 'train_loss': 2.0099681179086724, 'accuracy': 0.24927953890489912, 'macro_f1': 0.2166309276442026}, {'epoch': 2, 'train_loss': 1.732961068668881, 'accuracy': 0.24591738712776176, 

## After training

For the API to use this model, download `paddy_doctor_classifier.pth` from Drive and place it in the project's `models/` directory, replacing the interrupted CPU checkpoint. Download the validation JSON beside it. Verify the CUDA epoch log and final report before treating it as trained.

Scores are validation results from Paddy Doctor’s external image set. They do not establish disease prevalence, diagnostic accuracy in Sekinchan, or field-level performance. The dataset’s unlabeled test images are intentionally not used.